# Text Analysis of Martial Arts-related Subreddits

The purpose of this project is to analyse the differences and similarities between different subreddits existing around a common topic, in this case martial arts. At first, I will collect Reddit data, analyse TF-IDF scores, and attempt to classify the subreddits using k-means and Naive Bayes algorithms. Then, I will introduce networks to visualise connections between threaded comments and users. Finally, I will make use of embeddings to further the analysis.

## I/ Collecting Reddit Data and Exploring TF-IDF Results

In [13]:
# Setup autoreload
%load_ext autoreload
%autoreload 2

# Create README.md 
# pip3 install nbconvert
# jupyter nbconvert --execute --to markdown MartialArtsRedditAnalysis.ipynb
# then rename to README.md

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [14]:
from reddit_helper import *

Let's first collect  50 posts from the r/MuayThai, r/bjj, r/MMA, and r/Boxing subreddits, and display some of their initial characteristics. This is done using the `RedditScraper` class created in the `reddit_helper.py` file. 

In [16]:
# Example subreddits
subreddits = ['MuayThai', 'bjj', 'MMA', 'Boxing']

# Analysis parameters
MAX_TERMS = 1000
MIN_DOC_FREQ = 2
TOP_N = 10
LIMIT = 50
USERNAME = "matteolarrode"

# Initialize scraper
scraper = RedditScraper(
user_agent=f"SDS_textanalysis/1.0 (by /u/{USERNAME})"
)

# Analyze each subreddit independently
results = {}
submissions = {}

for subreddit in subreddits:
    print(f"\nAnalyzing r/{subreddit}...")

    # Collect posts
    submissions[subreddit] = scraper.get_subreddit_posts(subreddit, limit=LIMIT)

    # Analyze subreddit
    results[subreddit] = analyze_subreddit(
        submissions[subreddit],
        max_terms=MAX_TERMS,   # Maximum number of terms to keep
        min_doc_freq=MIN_DOC_FREQ,   # Term must appear in at least min_doc_freq documents
        top_n_terms=TOP_N # Number of top terms returned in result
    )

    # Print results for this subreddit
    print(f"\nVocabulary Statistics for r/{subreddit}:")
    print(f"Total words: {results[subreddit]['vocab_stats']['total_words']}")
    print(f"Unique words: {results[subreddit]['vocab_stats']['unique_words']}")
    print(f"Words appearing ≥{MIN_DOC_FREQ} times: {results[subreddit]['vocab_stats']['words_min_freq']}")
    print(f"Coverage by top {MAX_TERMS} words: {results[subreddit]['vocab_stats']['coverage_top_1000']:.2f}%")
    print(f"Matrix shape: {results[subreddit]['matrix_shape']}")
    print(f"Matrix sparsity: {results[subreddit]['matrix_sparsity']:.2f}%")

    print(f"\nTop {TOP_N} terms by TF-IDF score:")
    print(results[subreddit]['top_terms'][['term', 'score']].to_string())


Analyzing r/MuayThai...

Vocabulary Statistics for r/MuayThai:
Total words: 3211
Unique words: 996
Words appearing ≥2 times: 422
Coverage by top 1000 words: 100.00%
Matrix shape: (50, 238)
Matrix sparsity: 93.80%

Top 10 terms by TF-IDF score:
         term     score
62      fight  0.059222
97         im  0.053386
212  training  0.046307
129      muay  0.043908
199      thai  0.042514
139       one  0.042228
75     gloves  0.040059
32     clinch  0.038606
200  thailand  0.034214
68      first  0.033138

Analyzing r/bjj...

Vocabulary Statistics for r/bjj:
Total words: 5559
Unique words: 1453
Words appearing ≥2 times: 607
Coverage by top 1000 words: 91.85%
Matrix shape: (50, 403)
Matrix sparsity: 93.30%

Top 10 terms by TF-IDF score:
       term     score
41      bjj  0.051027
161      im  0.049453
123     get  0.045805
181    know  0.042780
15   anyone  0.041300
125      gi  0.039875
37   better  0.035872
35     belt  0.033848
137     gym  0.033784
195    like  0.033334

Analyzing r/M

Let's try to grasp a better understanding of the data structure of the `results` object. It is a dictionary with the following keys:


In [12]:
results['MuayThai'].keys()

dict_keys(['vocab_stats', 'freq_distribution', 'top_terms', 'vectorizer', 'matrix_shape', 'matrix_sparsity'])


The `results` object returns the top 5 terms. How would you access more than 5 terms? Expand the results to see 10. Consider what way is more general and flexible. How might you change the code so that there is a `TOP_N = 10` which is then passed through the code so that the results dictionary contains ten terms in the "top_terms" DataFrame rather than hard coding it in the method below?




2. **Store results**. Every time we run the code we query Reddit again. How can we store our data so that it is cached for another round? There are many approaches to this and among your group you may discover everything from 'just save the json' to 'DataFrame and then export to feather' to some who would ambitiously use MongoDB. Given this is a simple exercise for now, keep this step simple as you need it to be while still usable enough if you want to add more data.

3. **Plot keywords over time**. Expand your results to anywhere from 250 upwards (I would here cap at 500 max and think that the api might only return last 1000 but untested). Determine the top keywords using TFIDF. Then plot the frequency of these keywords over this time period for these results.   
4. **Table the most common URLs for stories**. Triangulate these plots with a table summarising the top news outlets for this sub in this time period. Notice the starter code to process this from the posts data that has been stored in a large `submissions` dictionary. Note, this code does not turn all the `json` into a DataFrame, but extracts only the URL column and processes that. It also uses a _regular expression_ to separate out the top level domain, which may or may not be the most robust.  
5. **Write a summary**. Solely for reflection at this point, write some intuitions that you discover with this exploration. 


In [5]:
# Data Exploration: 
submissions['MuayThai'][0] # Example post from the MuayThai subreddit

url_list = [post['url'] for post in submissions['MuayThai']]
url_df = pd.DataFrame(url_list, columns=['url'])
url_df['domain'] = url_df['url'].str.extract(r'(https?://[^/]+)')

url_df['domain'].value_counts().head(10)

domain
https://www.reddit.com     33
https://v.redd.it           8
https://i.redd.it           4
https://youtu.be            2
https://muay-ying.com       1
https://www.youtube.com     1
Name: count, dtype: int64

In [6]:
results['MuayThai'].keys()

dict_keys(['vocab_stats', 'freq_distribution', 'top_terms', 'vectorizer', 'matrix_shape', 'matrix_sparsity'])